In [ ]:
#!/usr/bin/env python3
"""
   Model Benchmarking, Dark Matter Substructure Classification              
   Trains multiple architectures under identical conditions                  
   for a fair experimentation comparison                                     

Models benchmarked (in order of complexity):
    ResNet-18          ~11.7M   (baseline CNN)
    ResNet-50          ~25.6M   (deeper residual)
    EfficientNet-B0    ~5.3M    (compound scaling, lightweight)
    EfficientNet-B4    ~19.3M   (compound scaling, mid-range)
    ConvNeXt V1 Tiny   ~28.6M   (modern ConvNet, no GRN)
    ConvNeXt V1 Base   ~88.6M   (modern ConvNet, no GRN)
    ConvNeXt V2 Base   ~88.7M   (GRN + FCMAE pretrain)
    ConvNeXt V2 Large  ~196.4M  (GRN + FCMAE pretrain) ← our final model

All models trained with:
    - Same 20 epochs (enough to distinguish architecture quality)
    - Same augmentation pipeline
    - Same AdamW + cosine schedule
    - Same batch size, label smoothing, Mixup
    - Same 90:10 train/val split
    - Same fixed test set (val/)
"""

import os, math, random, gc, warnings, json
from pathlib import Path
from datetime import datetime

os.environ["PYTORCH_CUDA_ALLOC_CONF"] = "expandable_segments:True"

import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
from torch.optim import AdamW

import timm
from timm.utils import ModelEma

from sklearn.metrics import roc_auc_score
from sklearn.model_selection import train_test_split

import albumentations as A
from albumentations.pytorch import ToTensorV2

import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt

from tqdm.auto import tqdm
warnings.filterwarnings('ignore')



# CONFIG
class CFG:
    DATA_ROOT  = "/kaggle/input/datasets/stellarquant/deeplensetask1/dataset"
    OUTPUT_DIR = "/kaggle/working/benchmark"

    #Fixed across ALL models for fair comparison 
    IMG_SIZE        = 224
    BATCH_SIZE      = 128
    NUM_WORKERS     = 4
    VAL_SPLIT       = 0.10
    EPOCHS          = 20        # enough to rank architectures reliably
    WARMUP_EPOCHS   = 2
    BASE_LR         = 3.125e-4  # = 6.25e-4 × 128/256 (linear scaling rule)
    WEIGHT_DECAY    = 0.05
    MIN_LR          = 1e-6
    LABEL_SMOOTHING = 0.1
    MIXUP_ALPHA     = 0.4
    EMA_DECAY       = 0.9999
    USE_BF16        = True
    GRAD_CLIP       = 1.0
    DROP_PATH       = 0.1

    # Dataset
    CLASS_NAMES = ['no_sub', 'subhalo', 'vortex']
    CLASS_DIRS  = {'no_sub': 'no', 'subhalo': 'sphere', 'vortex': 'vort'}
    PIXEL_MEAN  = 0.0615
    PIXEL_STD   = 0.1152
    SEED        = 42

    #  Models to benchmark 
    # (timm_name, display_name, pretrained)
    MODELS = [
        ("resnet18.a1_in1k",
         "ResNet-18", True),

        ("resnet50.a1_in1k",
         "ResNet-50", True),

        ("efficientnet_b0.ra_in1k",
         "EfficientNet-B0", True),

        ("efficientnet_b4.ra2_in1k",
         "EfficientNet-B4", True),

        ("convnext_tiny.fb_in22k_ft_in1k",
         "ConvNeXt V1 Tiny", True),

        ("convnext_base.fb_in22k_ft_in1k",
         "ConvNeXt V1 Base", True),

        ("convnextv2_base.fcmae_ft_in22k_in1k",
         "ConvNeXt V2 Base", True),

        ("convnextv2_large.fcmae_ft_in22k_in1k_384",
         "ConvNeXt V2 Large", True),
    ]


# SETUP
def seed_everything(seed=42):
    random.seed(seed); np.random.seed(seed)
    torch.manual_seed(seed); torch.cuda.manual_seed_all(seed)
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark     = False


# DATASET
class LensDataset(Dataset):
    def __init__(self, file_paths, labels, transform=None):
        self.file_paths = file_paths
        self.labels     = labels
        self.transform  = transform

    def __len__(self): return len(self.file_paths)

    def __getitem__(self, idx):
        img = np.load(self.file_paths[idx]).astype(np.float32)
        if img.ndim == 3: img = img[0]
        img_u8 = (img * 255).clip(0, 255).astype(np.uint8)
        if self.transform:
            return self.transform(image=img_u8)['image'], self.labels[idx]
        return torch.from_numpy(img_u8[None]).float() / 255.0, self.labels[idx]


def build_file_list(root: Path, split: str):
    paths, labels = [], []
    for i, cls in enumerate(CFG.CLASS_NAMES):
        cls_dir   = root / split / CFG.CLASS_DIRS[cls]
        npy_files = sorted(cls_dir.glob("*.npy"))
        paths.extend(str(p) for p in npy_files)
        labels.extend([i] * len(npy_files))
    return paths, labels


def get_train_transforms():
    return A.Compose([
        A.Resize(CFG.IMG_SIZE, CFG.IMG_SIZE, interpolation=2),
        A.HorizontalFlip(p=0.5),
        A.VerticalFlip(p=0.5),
        A.Rotate(limit=180, p=0.9, border_mode=0, value=0),
        A.RandomResizedCrop(
            size=(CFG.IMG_SIZE, CFG.IMG_SIZE),
            scale=(0.90, 1.00), ratio=(0.95, 1.05),
            interpolation=2, p=0.5,
        ),
        A.GaussNoise(var_limit=(0.65, 2.60), p=0.35),
        A.CoarseDropout(
            max_holes=4, max_height=18, max_width=18,
            min_holes=1, min_height=8, min_width=8,
            fill_value=0, p=0.20,
        ),
        A.Normalize(mean=[CFG.PIXEL_MEAN], std=[CFG.PIXEL_STD],
                    max_pixel_value=255.0),
        ToTensorV2(),
    ])


def get_val_transforms():
    return A.Compose([
        A.Resize(CFG.IMG_SIZE, CFG.IMG_SIZE, interpolation=2),
        A.Normalize(mean=[CFG.PIXEL_MEAN], std=[CFG.PIXEL_STD],
                    max_pixel_value=255.0),
        ToTensorV2(),
    ])


def replicate_channels(x): return x.repeat(1, 3, 1, 1)

def mixup_data(x, y, alpha=0.4):
    lam = np.random.beta(alpha, alpha) if alpha > 0 else 1.0
    idx = torch.randperm(x.size(0), device=x.device)
    return lam * x + (1 - lam) * x[idx], y, y[idx], lam

def mixup_loss(criterion, pred, y_a, y_b, lam):
    return lam * criterion(pred, y_a) + (1 - lam) * criterion(pred, y_b)


# SCHEDULER
def cosine_with_warmup(optimizer, warmup_epochs, total_epochs,
                        min_lr_ratio=0.01):
    def _fn(ep):
        if ep < warmup_epochs:
            return max(1e-6, (ep + 1) / warmup_epochs)
        prog = (ep - warmup_epochs) / max(1, total_epochs - warmup_epochs)
        return min_lr_ratio + 0.5*(1-min_lr_ratio)*(1+math.cos(math.pi*prog))
    return torch.optim.lr_scheduler.LambdaLR(optimizer, _fn)


# TRAIN & EVALUATE
def train_one_epoch(model, loader, optimizer, criterion,
                    scheduler, ema, device):
    model.train()
    dtype  = torch.bfloat16 if CFG.USE_BF16 else torch.float32
    losses = []
    for imgs, labels in loader:
        imgs   = replicate_channels(imgs).to(device, non_blocking=True)
        labels = labels.to(device, non_blocking=True)
        imgs, y_a, y_b, lam = mixup_data(imgs, labels, CFG.MIXUP_ALPHA)
        with torch.autocast('cuda', dtype=dtype):
            loss = mixup_loss(criterion, model(imgs), y_a, y_b, lam)
        optimizer.zero_grad(set_to_none=True)
        loss.backward()
        nn.utils.clip_grad_norm_(model.parameters(), CFG.GRAD_CLIP)
        optimizer.step()
        ema.update(model)
        losses.append(loss.item())
    scheduler.step()
    return float(np.mean(losses))


@torch.no_grad()
def evaluate(model, loader, device):
    model.eval()
    dtype = torch.bfloat16 if CFG.USE_BF16 else torch.float32
    all_probs, all_labels = [], []
    for imgs, labels in loader:
        imgs = replicate_channels(imgs).to(device, non_blocking=True)
        with torch.autocast('cuda', dtype=dtype):
            probs = F.softmax(model(imgs), dim=-1)
        all_probs.append(probs.cpu().float().numpy())
        all_labels.append(labels.numpy())
    probs  = np.concatenate(all_probs)
    labels = np.concatenate(all_labels)
    macro  = roc_auc_score(labels, probs, multi_class='ovr', average='macro')
    per_cls = {cls: roc_auc_score((labels==i).astype(int), probs[:,i])
               for i, cls in enumerate(CFG.CLASS_NAMES)}
    return macro, per_cls, probs, labels


# SINGLE MODEL BENCHMARK RUN
def run_model(timm_name, display_name, pretrained,
              train_loader, val_loader, test_loader, device):
    print(f"\n{'═'*65}")
    print(f"  {display_name}  ({timm_name})")
    print(f"{'═'*65}")

    gc.collect(); torch.cuda.empty_cache()

    #Build model 
    model = timm.create_model(
        timm_name, pretrained=pretrained,
        num_classes=3, in_chans=3,
        drop_path_rate=CFG.DROP_PATH,
    )
    # Grad checkpointing for large models only (saves VRAM)
    if hasattr(model, 'set_grad_checkpointing'):
        if sum(p.numel() for p in model.parameters()) > 50e6:
            model.set_grad_checkpointing(enable=True)

    n_params = sum(p.numel() for p in model.parameters())
    model    = model.to(device)
    ema      = ModelEma(model, decay=CFG.EMA_DECAY, device=device)
    criterion = nn.CrossEntropyLoss(label_smoothing=CFG.LABEL_SMOOTHING)
    print(f"  Params : {n_params/1e6:.1f}M  |  pretrained={pretrained}")

    #Optimizer & Scheduler
    optimizer = AdamW(model.parameters(),
                      lr=CFG.BASE_LR, weight_decay=CFG.WEIGHT_DECAY)
    scheduler = cosine_with_warmup(
        optimizer, CFG.WARMUP_EPOCHS, CFG.EPOCHS,
        min_lr_ratio=CFG.MIN_LR / CFG.BASE_LR,
    )

    # Training loop
    best_val_auc = 0.0
    val_auc_history = []

    for ep in range(CFG.EPOCHS):
        loss = train_one_epoch(model, train_loader, optimizer, criterion,
                               scheduler, ema, device)
        macro, per_cls, _, _ = evaluate(ema.ema, val_loader, device)
        val_auc_history.append(macro)
        if macro > best_val_auc:
            best_val_auc = macro
        cls_str = '  '.join(f'{k}={v:.4f}' for k, v in per_cls.items())
        print(f"  ep {ep+1:02d}/{CFG.EPOCHS}  loss={loss:.4f}  "
              f"val_auc={macro:.4f}  |  {cls_str}")

    #Test evaluation using best EMA weights 
    # (re-evaluate on test set with the model at its current EMA state)
    test_auc, test_per_cls, _, _ = evaluate(ema.ema, test_loader, device)

    print(f"\nTest Results ")
    print(f"Test Macro AUC : {test_auc:.4f}")
    for cls, val in test_per_cls.items():
        print(f"{cls:>10} : {val:.4f}")

  
    del model, ema, optimizer, scheduler, criterion
    gc.collect(); torch.cuda.empty_cache()

    return {
        'display_name':    display_name,
        'timm_name':       timm_name,
        'n_params_M':      round(n_params / 1e6, 1),
        'pretrained':      pretrained,
        'best_val_auc':    round(best_val_auc, 4),
        'test_macro_auc':  round(test_auc, 4),
        'test_per_cls':    {k: round(v, 4) for k, v in test_per_cls.items()},
        'val_auc_history': [round(v, 4) for v in val_auc_history],
    }


# PLOTS
def plot_comparison(results: list, save_dir: str):

    #1. AUC bar chart
    names  = [r['display_name'] for r in results]
    aucs   = [r['test_macro_auc'] for r in results]
    params = [r['n_params_M'] for r in results]

    # Colour by family
    def family_color(name):
        if 'ResNet'          in name: return '#e74c3c'
        if 'EfficientNet'    in name: return '#f39c12'
        if 'ConvNeXt V1'     in name: return '#3498db'
        if 'ConvNeXt V2'     in name: return '#2ecc71'
        return '#95a5a6'

    colors = [family_color(n) for n in names]

    fig, axes = plt.subplots(1, 2, figsize=(16, 6))
    fig.suptitle("Model Benchmark — Dark Matter Substructure Classification\n"
                 f"({CFG.EPOCHS} epochs, identical training conditions)",
                 fontsize=13, fontweight='bold')

    # Bar: test AUC
    ax = axes[0]
    bars = ax.barh(names, aucs, color=colors, alpha=0.85,
                   edgecolor='white', linewidth=1.2)
    ax.set_xlabel('Test Macro OvR AUC', fontsize=11)
    ax.set_title('Test AUC by Architecture', fontsize=11, fontweight='bold')
    ax.set_xlim(min(aucs) - 0.05, 1.005)
    ax.axvline(max(aucs), color='gold', lw=1.5, ls='--', label=f'Best={max(aucs):.4f}')
    ax.legend(fontsize=9); ax.grid(axis='x', alpha=0.25)
    for bar, val in zip(bars, aucs):
        ax.text(val + 0.001, bar.get_y() + bar.get_height()/2,
                f'{val:.4f}', va='center', fontsize=9, fontweight='bold')

    # Scatter: params vs AUC
    ax2 = axes[1]
    for r, col in zip(results, colors):
        ax2.scatter(r['n_params_M'], r['test_macro_auc'],
                    color=col, s=120, zorder=3, edgecolors='white', lw=1.5)
        ax2.annotate(r['display_name'],
                     (r['n_params_M'], r['test_macro_auc']),
                     textcoords='offset points', xytext=(6, 3),
                     fontsize=8)
    ax2.set_xlabel('Parameters (M)', fontsize=11)
    ax2.set_ylabel('Test Macro OvR AUC', fontsize=11)
    ax2.set_title('AUC vs Model Size', fontsize=11, fontweight='bold')
    ax2.grid(alpha=0.25)

    # Legend for families
    from matplotlib.patches import Patch
    legend_elements = [
        Patch(facecolor='#e74c3c', label='ResNet'),
        Patch(facecolor='#f39c12', label='EfficientNet'),
        Patch(facecolor='#3498db', label='ConvNeXt V1'),
        Patch(facecolor='#2ecc71', label='ConvNeXt V2'),
    ]
    ax2.legend(handles=legend_elements, fontsize=9)

    plt.tight_layout()
    path = f"{save_dir}/benchmark_comparison.png"
    plt.savefig(path, dpi=150, bbox_inches='tight')
    plt.close()
    print(f"  Saved → {path}")

    #2. Val AUC training curves
    fig, ax = plt.subplots(figsize=(13, 6))
    for r, col in zip(results, colors):
        ax.plot(range(1, CFG.EPOCHS+1), r['val_auc_history'],
                label=f"{r['display_name']} ({r['test_macro_auc']:.4f})",
                color=col, lw=2, alpha=0.85)
    ax.set_xlabel('Epoch'); ax.set_ylabel('Val Macro OvR AUC')
    ax.set_title(f'Validation AUC Curves — All Models  ({CFG.EPOCHS} epochs)',
                 fontsize=12, fontweight='bold')
    ax.legend(fontsize=9, loc='lower right'); ax.grid(alpha=0.2)
    plt.tight_layout()
    path = f"{save_dir}/benchmark_curves.png"
    plt.savefig(path, dpi=150, bbox_inches='tight')
    plt.close()
    print(f"  Saved → {path}")

    # 3. Per-class AUC heatmap
    cls_data = np.array([[r['test_per_cls'][c] for c in CFG.CLASS_NAMES]
                          for r in results])
    fig, ax = plt.subplots(figsize=(8, len(results) * 0.65 + 1.5))
    im = ax.imshow(cls_data, cmap='RdYlGn', vmin=0.85, vmax=1.0,
                   aspect='auto')
    plt.colorbar(im, ax=ax, label='AUC')
    ax.set_xticks(range(3)); ax.set_xticklabels(CFG.CLASS_NAMES, fontsize=11)
    ax.set_yticks(range(len(results)))
    ax.set_yticklabels([r['display_name'] for r in results], fontsize=10)
    ax.set_title('Per-Class AUC Heatmap\n(darker green = higher AUC)',
                 fontsize=11, fontweight='bold')
    for i in range(len(results)):
        for j in range(3):
            ax.text(j, i, f'{cls_data[i,j]:.4f}', ha='center', va='center',
                    fontsize=9, fontweight='bold',
                    color='white' if cls_data[i,j] < 0.92 else 'black')
    plt.tight_layout()
    path = f"{save_dir}/benchmark_per_class_heatmap.png"
    plt.savefig(path, dpi=150, bbox_inches='tight')
    plt.close()
    print(f"  Saved → {path}")


def print_summary_table(results: list):
    print(f"\n{'═'*75}")
    print(f"  BENCHMARK SUMMARY  ({CFG.EPOCHS} epochs, identical conditions)")
    print(f"{'═'*75}")
    header = (f"  {'Model':<22}  {'Params':>8}  {'Val AUC':>9}  "
              f"{'Test AUC':>9}  {'no_sub':>8}  {'subhalo':>8}  {'vortex':>8}")
    print(header)
    print("  " + "─" * (len(header) - 2))
    for r in sorted(results, key=lambda x: x['test_macro_auc'], reverse=True):
        pc = r['test_per_cls']
        marker = " ◀ best" if r == max(results, key=lambda x: x['test_macro_auc']) else ""
        print(f"  {r['display_name']:<22}  {r['n_params_M']:>7.1f}M  "
              f"{r['best_val_auc']:>9.4f}  {r['test_macro_auc']:>9.4f}  "
              f"{pc['no_sub']:>8.4f}  {pc['subhalo']:>8.4f}  "
              f"{pc['vortex']:>8.4f}{marker}")
    print(f"{'═'*75}\n")


# MAIN
def main():
    seed_everything(CFG.SEED)
    os.makedirs(CFG.OUTPUT_DIR, exist_ok=True)
    device = torch.device('cuda')
    print(f"GPU  : {torch.cuda.get_device_name(0)}")
    print(f"Benchmarking {len(CFG.MODELS)} models × {CFG.EPOCHS} epochs each\n")

    # ── Build shared data loaders (same split for ALL models) ─────────────
    data_root = Path(CFG.DATA_ROOT)
    train_paths, train_labels = build_file_list(data_root, 'train')
    test_paths,  test_labels  = build_file_list(data_root, 'val')

    tr_paths, val_paths, tr_labels, val_labels = train_test_split(
        train_paths, train_labels,
        test_size=CFG.VAL_SPLIT, stratify=train_labels,
        random_state=CFG.SEED,
    )
    print(f"Train: {len(tr_paths):,}  Val: {len(val_paths):,}  "
          f"Test: {len(test_paths):,}\n")

    train_loader = DataLoader(
        LensDataset(tr_paths,   tr_labels,   get_train_transforms()),
        batch_size=CFG.BATCH_SIZE, shuffle=True,
        num_workers=CFG.NUM_WORKERS, pin_memory=True,
        drop_last=True, persistent_workers=True,
    )
    val_loader = DataLoader(
        LensDataset(val_paths,  val_labels,  get_val_transforms()),
        batch_size=CFG.BATCH_SIZE * 2, shuffle=False,
        num_workers=CFG.NUM_WORKERS, pin_memory=True, persistent_workers=True,
    )
    test_loader = DataLoader(
        LensDataset(test_paths, test_labels, get_val_transforms()),
        batch_size=CFG.BATCH_SIZE * 2, shuffle=False,
        num_workers=CFG.NUM_WORKERS, pin_memory=True, persistent_workers=True,
    )

    #Run all models
    results = []
    for timm_name, display_name, pretrained in CFG.MODELS:
        try:
            result = run_model(
                timm_name, display_name, pretrained,
                train_loader, val_loader, test_loader, device,
            )
            results.append(result)
            # Save results incrementally, safe against session interruption
            with open(f"{CFG.OUTPUT_DIR}/results.json", 'w') as f:
                json.dump(results, f, indent=2)
        except Exception as e:
            print(f"\n  ✗ {display_name} FAILED: {e}")
            results.append({
                'display_name': display_name,
                'timm_name':    timm_name,
                'error':        str(e),
            })

    # Summary & plots
    valid = [r for r in results if 'test_macro_auc' in r]
    print_summary_table(valid)
    plot_comparison(valid, CFG.OUTPUT_DIR)

    # Save final JSON
    with open(f"{CFG.OUTPUT_DIR}/results.json", 'w') as f:
        json.dump(results, f, indent=2)
    print(f"\n  Results JSON → {CFG.OUTPUT_DIR}/results.json")
    print(f"  Plots        → {CFG.OUTPUT_DIR}/")


if __name__ == '__main__':
    main()

GPU  : NVIDIA H100 80GB HBM3
Benchmarking 8 models × 20 epochs each

Train: 27,000  Val: 3,000  Test: 7,500


═════════════════════════════════════════════════════════════════
  ResNet-18  (resnet18.a1_in1k)
═════════════════════════════════════════════════════════════════


model.safetensors:   0%|          | 0.00/46.8M [00:00<?, ?B/s]

  Params : 11.2M  |  pretrained=True
  ep 01/20  loss=1.1019  val_auc=0.5017  |  no_sub=0.5193  subhalo=0.5095  vortex=0.4762
  ep 02/20  loss=1.0990  val_auc=0.5032  |  no_sub=0.5211  subhalo=0.5109  vortex=0.4777
  ep 03/20  loss=1.0935  val_auc=0.5050  |  no_sub=0.5236  subhalo=0.5134  vortex=0.4781
  ep 04/20  loss=1.0685  val_auc=0.5066  |  no_sub=0.5235  subhalo=0.5187  vortex=0.4776
  ep 05/20  loss=1.0405  val_auc=0.5085  |  no_sub=0.5247  subhalo=0.5230  vortex=0.4777
  ep 06/20  loss=1.0097  val_auc=0.5091  |  no_sub=0.5242  subhalo=0.5266  vortex=0.4765
  ep 07/20  loss=0.9936  val_auc=0.5122  |  no_sub=0.5265  subhalo=0.5320  vortex=0.4782
  ep 08/20  loss=0.9758  val_auc=0.5160  |  no_sub=0.5303  subhalo=0.5379  vortex=0.4797
  ep 09/20  loss=0.9735  val_auc=0.5183  |  no_sub=0.5318  subhalo=0.5419  vortex=0.4813
  ep 10/20  loss=0.9478  val_auc=0.5210  |  no_sub=0.5335  subhalo=0.5462  vortex=0.4833
  ep 11/20  loss=0.9472  val_auc=0.5214  |  no_sub=0.5320  subhalo=0.5462

model.safetensors:   0%|          | 0.00/102M [00:00<?, ?B/s]

  Params : 23.5M  |  pretrained=True
  ep 01/20  loss=1.0998  val_auc=0.4947  |  no_sub=0.4715  subhalo=0.5114  vortex=0.5012
  ep 02/20  loss=1.0993  val_auc=0.4948  |  no_sub=0.4711  subhalo=0.5093  vortex=0.5038
  ep 03/20  loss=1.0956  val_auc=0.4953  |  no_sub=0.4720  subhalo=0.5093  vortex=0.5045
  ep 04/20  loss=1.0729  val_auc=0.4975  |  no_sub=0.4743  subhalo=0.5130  vortex=0.5053
  ep 05/20  loss=1.0163  val_auc=0.4978  |  no_sub=0.4756  subhalo=0.5145  vortex=0.5034
  ep 06/20  loss=0.9784  val_auc=0.5005  |  no_sub=0.4863  subhalo=0.5083  vortex=0.5070
  ep 07/20  loss=0.9552  val_auc=0.5040  |  no_sub=0.4932  subhalo=0.5139  vortex=0.5048
  ep 08/20  loss=0.9434  val_auc=0.5068  |  no_sub=0.4973  subhalo=0.5148  vortex=0.5083
  ep 09/20  loss=0.9209  val_auc=0.5157  |  no_sub=0.5125  subhalo=0.5192  vortex=0.5153
  ep 10/20  loss=0.9236  val_auc=0.5168  |  no_sub=0.5204  subhalo=0.5180  vortex=0.5119
  ep 11/20  loss=0.9127  val_auc=0.5272  |  no_sub=0.5417  subhalo=0.5204

...

  Params : 87.6M  |  pretrained=True
  ep 01/20  loss=1.0917  val_auc=0.5063  |  no_sub=0.5228  subhalo=0.5305  vortex=0.4655
  ep 02/20  loss=0.9346  val_auc=0.5098  |  no_sub=0.5306  subhalo=0.5419  vortex=0.4570
  ep 03/20  loss=0.9169  val_auc=0.5211  |  no_sub=0.5552  subhalo=0.5501  vortex=0.4580
  ep 04/20  loss=0.8726  val_auc=0.5346  |  no_sub=0.5750  subhalo=0.5576  vortex=0.4711
  ep 05/20  loss=0.8767  val_auc=0.5531  |  no_sub=0.5937  subhalo=0.5678  vortex=0.4976
  ep 06/20  loss=0.8612  val_auc=0.5753  |  no_sub=0.6163  subhalo=0.5808  vortex=0.5289
  ep 07/20  loss=0.8660  val_auc=0.5996  |  no_sub=0.6418  subhalo=0.5981  vortex=0.5589
  ep 08/20  loss=0.8480  val_auc=0.6250  |  no_sub=0.6695  subhalo=0.6179  vortex=0.5876
  ep 09/20  loss=0.8413  val_auc=0.6497  |  no_sub=0.6976  subhalo=0.6383  vortex=0.6133
  ep 10/20  loss=0.8478  val_auc=0.6730  |  no_sub=0.7240  subhalo=0.6579  vortex=0.6372
  ep 11/20  loss=0.8336  val_auc=0.6948  |  no_sub=0.7490  subhalo=0.6775

model.safetensors:   0%|          | 0.00/355M [00:00<?, ?B/s]

  Params : 87.7M  |  pretrained=True
  ep 01/20  loss=1.1136  val_auc=0.4905  |  no_sub=0.4756  subhalo=0.4944  vortex=0.5013
  ep 02/20  loss=1.1025  val_auc=0.4976  |  no_sub=0.4930  subhalo=0.5016  vortex=0.4982
  ep 03/20  loss=1.1012  val_auc=0.5018  |  no_sub=0.4997  subhalo=0.5081  vortex=0.4975
  ep 04/20  loss=1.1010  val_auc=0.5028  |  no_sub=0.5019  subhalo=0.5128  vortex=0.4938
  ep 05/20  loss=1.1003  val_auc=0.5035  |  no_sub=0.5034  subhalo=0.5201  vortex=0.4870
  ep 06/20  loss=1.1004  val_auc=0.5098  |  no_sub=0.5161  subhalo=0.5273  vortex=0.4861
  ep 07/20  loss=1.1000  val_auc=0.5161  |  no_sub=0.5333  subhalo=0.5252  vortex=0.4898
  ep 08/20  loss=1.1000  val_auc=0.5176  |  no_sub=0.5349  subhalo=0.5200  vortex=0.4980
  ep 09/20  loss=1.0995  val_auc=0.5120  |  no_sub=0.5318  subhalo=0.5053  vortex=0.4990
  ep 10/20  loss=1.0993  val_auc=0.5187  |  no_sub=0.5345  subhalo=0.5051  vortex=0.5163
  ep 11/20  loss=1.0995  val_auc=0.5251  |  no_sub=0.5330  subhalo=0.5166

model.safetensors:   0%|          | 0.00/792M [00:00<?, ?B/s]

  Params : 196.4M  |  pretrained=True
  ep 01/20  loss=1.1192  val_auc=0.5345  |  no_sub=0.5267  subhalo=0.5338  vortex=0.5429
  ep 02/20  loss=1.1034  val_auc=0.5270  |  no_sub=0.5302  subhalo=0.5248  vortex=0.5261
  ep 03/20  loss=1.1009  val_auc=0.5121  |  no_sub=0.5128  subhalo=0.5090  vortex=0.5145
  ep 04/20  loss=1.1007  val_auc=0.5017  |  no_sub=0.4984  subhalo=0.5053  vortex=0.5015
  ep 05/20  loss=1.1007  val_auc=0.4964  |  no_sub=0.5168  subhalo=0.4810  vortex=0.4913
  ep 06/20  loss=1.1008  val_auc=0.4970  |  no_sub=0.5125  subhalo=0.4836  vortex=0.4948
  ep 07/20  loss=1.1005  val_auc=0.4905  |  no_sub=0.4953  subhalo=0.4760  vortex=0.5003
  ep 08/20  loss=1.1002  val_auc=0.5040  |  no_sub=0.4981  subhalo=0.5098  vortex=0.5041
  ep 09/20  loss=1.0996  val_auc=0.5146  |  no_sub=0.5165  subhalo=0.5195  vortex=0.5078
  ep 10/20  loss=1.0999  val_auc=0.5149  |  no_sub=0.5192  subhalo=0.5219  vortex=0.5036
  ep 11/20  loss=1.0993  val_auc=0.5070  |  no_sub=0.4952  subhalo=0.515